# Logistic Regression — Newton Method Formulation

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \{0,1\}
$$

is the binary class label.

If a bias term is included, the design matrix becomes

$$
X \in \mathbb{R}^{N \times (D+1)}
$$

and the parameter vector is

$$
w \in \mathbb{R}^{D+1}
$$

---

# 2. Logistic Regression Model

Logistic regression models the probability of class \(1\) as

$$
P(y=1|x) = \sigma(z)
$$

where

$$
z = x^T w
$$

and

$$
\sigma(z) =
\frac{1}{1+e^{-z}}
$$

is the **sigmoid function**.

Thus the predicted probability for sample \(i\) is

$$
p_i = \sigma(x_i^T w)
$$

---

# 3. Likelihood Function

Assuming independent observations, the likelihood of the dataset is

$$
L(w) =
\prod_{i=1}^{N}
p_i^{y_i}(1-p_i)^{1-y_i}
$$

Taking the logarithm gives the **log-likelihood**

$$
\ell(w) =
\sum_{i=1}^{N}
\left[
y_i \log p_i +
(1-y_i)\log(1-p_i)
\right]
$$

Logistic regression estimates parameters by **maximizing the log-likelihood**.

Equivalently, we minimize the **negative log-likelihood (cross-entropy loss)**

$$
L(w) =
-\sum_{i=1}^{N}
\left[
y_i \log p_i +
(1-y_i)\log(1-p_i)
\right]
$$

---

# 4. Gradient of the Loss Function

Let

$$
p = \sigma(Xw)
$$

The gradient of the loss function is

$$
\nabla L(w) =
X^T (p-y)
$$

where

$$
p =
\begin{bmatrix}
p_1 \\
p_2 \\
\vdots \\
p_N
\end{bmatrix}
$$

and

$$
y =
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

---

# 5. Hessian Matrix

The Hessian matrix is the second derivative of the loss function.

It can be written as

$$
H =
X^T R X
$$

where

$$
R =
\text{diag}(p_i(1-p_i))
$$

is a diagonal matrix containing the **variance of each Bernoulli observation**.

Thus

$$
R =
\begin{bmatrix}
p_1(1-p_1) & 0 & \dots & 0 \\
0 & p_2(1-p_2) & \dots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \dots & p_N(1-p_N)
\end{bmatrix}
$$

---

# 6. Newton's Method Update

Newton's method uses second-order information to update parameters.

The update rule is

$$
w_{t+1}
=
w_t - H^{-1} \nabla L(w_t)
$$

Substituting the expressions for the gradient and Hessian gives

$$
w_{t+1}
=
w_t
-
(X^T R X)^{-1}
X^T (p-y)
$$

This procedure is repeated until convergence. This approach is also known as **Iteratively Reweighted Least Squares (IRLS)** and typically converges much faster than gradient descent.

---

# 7. Convergence Criterion

The algorithm stops when the gradient becomes sufficiently small:

$$
\|\nabla L(w)\| < \text{tol}
$$

where **tol** is a small tolerance value.

---

# 8. Prediction Rule

After training, probabilities are computed as

$$
p = \sigma(Xw)
$$

The predicted class label is obtained using the threshold

$$
\hat{y} =
\begin{cases}
1 & p \ge 0.5 \\
0 & p < 0.5
\end{cases}
$$

---

In [1]:
class LogisticRegressionNewton:
    def __init__(self,num_iterations=10000 , tol= 1e-8 , add_bias=True):
        self.num_iterations = num_iterations
        self.tol = tol
        self.add_bias = add_bias

        self.weights = None

    def sigmoid(self,X):
        X = np.asarray(X)
        out = np.empty_like(X,dtype=float)
        positive = X>=0
        negative = ~ positive

        out[positive] = 1/ (1+ np.exp(-X[positive]))
        out[negative] = np.exp(X[negative])/ (1+ np.exp(X[negative]))

        return out

    def fit(self,X,y):

        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1)

        if X.ndim == 1:
            X = X.reshape(-1,1)

        N, D = X.shape

        if self.add_bias :
            X = np.column_stack((np.ones(N),X))
            D +=1

        
        self.weights = np.zeros(D)

        hess_tol = 1e-6*np.eye(D)

        from scipy.linalg import solve

        XT = X.T
        XTy = XT @ y

        for i in range(self.num_iterations):


                        
            Z = X @ self.weights
            
            p_n = self.sigmoid(Z)
            
            eps = 1e-12
            p_n[p_n < eps] = eps
            p_n[p_n > 1-eps] = 1-eps
            
            grad_loss = XT @ p_n - XTy

            R = p_n * (1-p_n)
            XR = X * R[:, None]

            Hess = XT @ XR + hess_tol
            
            self.weights = self.weights -  np.linalg.inv(Hess) @ grad_loss
            if i > 0 and np.linalg.norm(grad_loss)  < self.tol :        
                print(f"Converged in iteration: {i}")
                break
        print(f"Model Weights : {self.weights}")

        return 


    def predict(self,X):
        X = np.asarray(X)
       

        if X.ndim == 1:
            X = X.reshape(-1,1)

        N = len(X)

        if self.add_bias :
            X = np.column_stack((np.ones(N),X))
           
            
        Z = X @ self.weights
        p_n = self.sigmoid(Z)

        return np.where(p_n>=0.5,1,0)
       

## 1. Logistic Regression (Without Bias)

Assume the true weight vector is

$$
w =
\begin{bmatrix}
2 \\
-1.5
\end{bmatrix}
$$

Data is generated using these weights and the logistic model.


Estimated weights:

$$
\hat{w} \approx
\begin{bmatrix}
2 \\
-1.5
\end{bmatrix}
$$





In [2]:
import numpy as np

np.random.seed(42)

N = 10000
D = 2

X = np.random.randn(N, D)
w_true = np.array([2.0, -1.5])
z = X @ w_true

p = 1 / (1 + np.exp(-z))

y = np.random.binomial(1, p)

print("True weights:", w_true)

True weights: [ 2.  -1.5]


In [3]:
model = LogisticRegressionNewton(add_bias=False, tol=1e-10)
model.fit(X, y)

Converged in iteration: 7
Model Weights : [ 2.08576572 -1.56445267]


---


## 2. Logistic Regression (With Bias)

Assume the true parameters are

$$
w =
\begin{bmatrix}
0.7 \\
1.5 \\
-2
\end{bmatrix}
$$

where the first element represents the bias.



Estimated parameters:

$$
\hat{w} \approx
\begin{bmatrix}
0.7 \\
1.5 \\
-2
\end{bmatrix}
$$

In [4]:
np.random.seed(42)

N = 10000
D = 2

X = np.random.randn(N, D)

bias_true = 0.7
w_true = np.array([1.5, -2.0])

z = bias_true + X @ w_true

p = 1 / (1 + np.exp(-z))

y = np.random.binomial(1, p)

print("True bias:", bias_true)
print("True weights:", w_true)

True bias: 0.7
True weights: [ 1.5 -2. ]


In [5]:
model = LogisticRegressionNewton(add_bias=True, tol=1e-10)
model.fit(X, y)

Converged in iteration: 7
Model Weights : [ 0.70269611  1.52275368 -2.0368958 ]


---